# 07 · Chunking 基础

> 分块决定检索的最小单位。chunk 太碎会召回“半句话”，太整会让语义被稀释。这是 RAG 里最需要手感的环节。

**本文件覆盖知识点**：Fixed-size / Character / Token / Sentence / Recursive Character Text Splitter；chunk_size / chunk_overlap / token 数量

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. chunk_size 与 overlap

- **chunk_size（块大小）**：每块切多长。
  - 太小：信息碎片化，一个 chunk 讲不清一件事；
  - 太大：一个向量平均了太多内容，检索“找不到那一段细节”。
- **chunk_overlap（重叠）**：相邻块重叠多少字符。防止关键句恰好落在切缝里被一分为二。

> 度量单位可以是“字符 / 词 / token”。中文建议看 token（模型按 token 计费、按 token 塞上下文）。
> 经验参考：200~500 字、overlap 为 10%~20%；具体数值要做实验评测（第 34 课）。

In [ ]:
# 演示 chunk_size / overlap 的机制（字符级）
from pathlib import Path

text = Path('data/向量数据库.md').read_text(encoding='utf-8')

def fixed_size_split(text, chunk_size=120, overlap=30):
    """固定步长切分：step = chunk_size - overlap"""
    step = chunk_size - overlap
    return [text[i:i + chunk_size] for i in range(0, len(text), step)]

chunks = fixed_size_split(text, 120, 30)
print('原始长度:', len(text), '| 切出块数:', len(chunks))
for i, c in enumerate(chunks[:4]):
    print(f'\n[chunk {i}] (len={len(c)})')
    print(c)
# 观察: 固定切分常把段落/句子拦腰截断 —— 这是它的缺陷

In [ ]:
# 知识点·真调说明：chunk_size 粒度 —— 同一问句，喂「被切缝砍成半句的块」vs「完整语义单元」，看回答质量差多少
# 模拟一次检索只命中某个 chunk：若切分把“方案名”与“它的解释”割到两块，命中的这块就缺了关键名词。
print('① chunk 太碎 / 切缝落在句中 —— 模型只拿到半个事实，回答残缺')
_llm_live(
    prompt="""系统用什么手段拦截不存在的 key？请只依据下面给出的资料回答；资料里没写就明确说“资料未提及”。
资料：「…内存就能判断 key 是否可能存在，误判率可控制在 1% 以下。」""",
    system='你是客服系统的问答助手，只能依据给定资料作答，禁止脑补资料里没有的内容。',
    fallback="""资料未提及拦截手段的名字，只描述了一个方案的特征（用很小内存判断 key 是否存在、误判率 1% 以下）——因为方案名“布隆过滤器”恰好被切到相邻的另一块里去了。""",
    temperature=0.1,
)
print()
print('② chunk 完整（方案名与解释同块）—— 同一问句直接答出')
_llm_live(
    prompt="""系统用什么手段拦截不存在的 key？请只依据下面给出的资料回答。
资料：「团队决定采用布隆过滤器拦截不存在的 key：它只需要很小的内存就能判断 key 是否可能存在，误判率可控制在 1% 以下。」""",
    system='你是客服系统的问答助手，只能依据给定资料作答，禁止脑补资料里没有的内容。',
    fallback="""系统采用布隆过滤器拦截不存在的 key：只需很小内存即可判断 key 是否可能存在，误判率可控制在 1% 以下。""",
    temperature=0.1,
)
print()
print('→ 同一问句，只因“命中的那一块”缺了方案名，回答就从能答变成答不出。')
print('  这正是 chunk 太碎 / 切缝落在句中导致的检索失败；合适的 chunk_size、overlap 兜底、按句切，都是为了把这种“缝”错开。')

## 2. 五种基础切分器对比

| 方法 | 怎么切 | 优点 | 缺点 |
|------|--------|------|------|
| **Character Split** | 按固定字符数 | 简单可控 | 会切断句子 |
| **Fixed-size Token Split** | 按 token 数 | 与模型口径一致 | 中文词边界难精确 |
| **Sentence Split** | 按句子边界 | 语义完整 | 长句超限、块大小不均 |
| **Recursive Character** | 优先段落/行/句/字逐级切 | 兼顾语义与大小，最常用 | 需要调分隔符与长度 |
| 固定大小 Fixed-size | 字符串硬切 | 最简 | 语义最差 |

> Recursive Character Text Splitter（LangChain 标配）是目前生产最常用的基础切分器，原理就在下一段手写。

In [ ]:
# 手写一个精简版 Recursive Splitter：优先在“自然边界”切
def recursive_split(text, chunk_size=120, overlap=30):
    """递归切分: 优先按段落/换行/句子边界，最后才硬切"""
    seps = ['\n\n', '\n', '。', '！', '？', '；', ' ']  # 分隔符优先级

    def _split(t, ss):
        if len(t) <= chunk_size:
            return [t] if t else []
        sep = ss[0] if ss else None
        if sep and sep in t:
            pieces = [p + sep for p in t.split(sep)[:-1]] + [t.split(sep)[-1]]
            out, buf = [], ''
            for p in pieces:                      # 尽量把相邻片段拼满一块
                if len(buf) + len(p) <= chunk_size:
                    buf += p
                else:
                    if buf:
                        out.append(buf)
                    buf = p
            if buf:
                out.append(buf)
            return out
        return _split(t, ss[1:]) if len(ss) > 1 else [t[i:i + chunk_size] for i in range(0, len(t), chunk_size)]

    raw = _split(text, seps)
    result = [raw[0]]
    for i in range(1, len(raw)):                  # 叠加 overlap
        result.append(raw[i - 1][-overlap:] + raw[i])
    return result

chunks2 = recursive_split(text)
print('递归切出块数:', len(chunks2))
for i, c in enumerate(chunks2[:3]):
    print(f'\n[chunk {i}]')
    print(c)

In [ ]:
# 知识点·真调说明：Recursive 按自然边界切 —— 让模型当“切块质检员”，判定哪块语义自洽、哪块像被腰斩
# 固定硬切常把一句话劈成两半；递归切分尽量在句/段边界收尾。让 LLM 用“能否独立成段”来验收这个差别。
_llm_live(
    prompt="""下面两个片段都来自同一篇《数据库连接池调优》，是两种切分器产出的块。请判断哪个片段“语义更完整、更像一个能独立入库的知识单元”，只回答 A 或 B，并用一句话说明理由。
A：「…数据库连接池不是越大越好。过大时会占用过多内存，并导致连接创建与销毁的开销上升，还可能拖慢应用启动——这些都要在配」
B：「…数据库连接池不是越大越好。过大时连接创建与销毁的开销都会上升。建议先用默认值上线，再按压测逐步调大，并观察连接获取耗时。」""",
    system='你是 RAG 数据质量评审，只需回答“选 A/B + 一句理由”。',
    fallback="""选 B。A 在“这些都要在配”处戛然而止，像被拦腰截断；B 在完整的句子和论点处收尾，能独立成一条知识。""",
    temperature=0.1,
)
print('→ 模型能当“切块质检员”，说明“语义完整”可被判别——这正是 Recursive Splitter 优先在段落/句子边界收尾、而非按字符硬切的原因。')

## 小结

- 两个核心旋钮：chunk_size（粒度）与 chunk_overlap（防切断）；
- 按自然边界（段落 → 句子）切，语义最完整，即 Recursive Splitter 的思想；
- chunk 大小需结合评测数据调（第 34 课）。

基础切分不区分“这句话讲什么”。要让每个 chunk 语义内聚，需要语义化、层级化切分，见下一课（08）。